In [ ]:
import os
import sys
import seaborn as sns
from scipy.stats import norm
import pymc as pm
import pymc_bart as pmb
import pandas as pd
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
rng = np.random.default_rng(123)

output_folder = "ISYE6420/Project/"

In [ ]:
# Import configurations
dx_name = "asthma"
version = "v7_040226"

cohort = pd.read_csv(f"ISYE6420\\Project\\data\\asthma_weighted_cohort.csv")
print(len(cohort))

y = cohort["outcome"].values
treatment = cohort["exposure"].values
follow_up_time = cohort["follow_up_time"].values
# Convert to binary indicator for outcome
y_data = (y > 0).astype(int)


In [ ]:
X = cohort.drop(columns=["patid", "outcome", "exposure", "follow_up_time", "weight", "ps_score"])
# convert True False columns to int
X = X.astype(int)
# Logit transform the propensity scores for better modeling
propensity_score = cohort["ps_score"]
eps = 1e-6
ps_values = np.clip(propensity_score, eps, 1 - eps)
logit_ps = np.log(ps_values / (1 - ps_values))
X['logit_ps'] = logit_ps
X_mu = X.values

prognostic_model_config = {
    "n_trees": 100,
    "beta_prior": 2,
    "alpha_prior": 0.90,
}

In [ ]:
num_exac_pats = cohort[cohort['outcome'] > 0]['patid'].nunique()/ cohort['patid'].nunique()
print(f"Proportion of patients with at least one event: {num_exac_pats:.2%}")

In [ ]:
# Treatment indicator as int
T = treatment.astype(int)

# Feature matrix for tau: same covariates as mu, but add treatment indicator instead of propensity score
X["T"] = T
X_tau = X.values.astype(float)

tau_model_config = {
    "n_trees": 25,       # stronger regularisation toward zero for treatment effects
    "beta_prior": 3,     # more shrinkage towards homogenenous treatment effects
    "alpha_prior": 0.25,
}


In [ ]:
import numpy as np
# Assuming the standard xbart python wrapper is installed (pip install xbart)
from xbart import XBCF 

# 1. Ensure Data is in standard format (C-contiguous numpy arrays)
# X_mu: Prognostic covariates (N x P)
# X_tau: Treatment covariates (N x P)
# T: Treatment indicator array of 0s and 1s (N,)
# y_data: Binary outcome array of 0s and 1s (N,)
# p_hat: Estimated propensity scores (N,)

y_obs_binary = np.where(y_data > 0, 1, 0).astype(np.int32)
T_binary = T.astype(np.int32)

# 2. Define the Model Configurations (Mapping your exact requested priors)
xbcf_model = XBCF(
    # --- Prognostic Forest (mu) ---
    num_trees_pr=100,        # Scaled down to prevent probit saturation
    alpha_pr=0.95,           # High probability of splitting
    beta_pr=2.0,             # Standard depth penalty
    
    # --- Treatment Forest (tau) ---
    num_trees_trt=20,        # Highly restricted weak learners
    alpha_trt=0.25,          # Low probability of splitting
    beta_trt=3.0,            # Aggressive depth penalty to shrink toward homogeneity
    
    # --- Probit/Binary Configuration ---
    model="probit",          # Tells XBCF to use the latent normal data augmentation
    
    # --- Sampling Configuration ---
    num_sweeps=1000,         # Equivalent to standard draws
    burnin=200,              # XBCF burns in extremely fast, 200 is usually plenty
    
    # --- Optional but recommended for stability ---
    mtry_pr=X_mu.shape[1],   # Use all features for splits (or set a fraction for Random Forest style)
    mtry_trt=X_tau.shape[1]
)

# 3. Fit the Joint Model
# XBCF fits everything in one generative sweep, managing the non-conjugate updates internally
xbcf_model.fit(
    x_con=X_mu,              # Prognostic variables
    x_mod=X_tau,             # Treatment modifying variables
    z=T_binary,              # Treatment indicator
    y=y_obs_binary,          # Binary outcome (0 or 1)
    pihat=p_hat              # Propensity scores
)

# 4. Extracting the Latent Treatment Effects
# The tau predictions are on the PROBIT (latent) scale, not the absolute risk scale.
# Shape will be (num_sweeps - burnin, N)
tau_posterior_latent = xbcf_model.predict_trt(X_tau)

# Calculate the posterior mean CATT on the latent scale
catt_latent_mean = tau_posterior_latent.mean(axis=0)

print(f"Sampling complete. XBCF processed {tau_posterior_latent.shape[0]} sweeps.")